## Shadow model via interpolation (RBF Kernel) on Sparse grid

For QML model with 6 features, interpolation on sparse grid with RBF kernel 

This is more for proof of concept to check if there is a benefit of using a sparse grid; the interpolation is not efficient (and technically not regression free)

We load here the sparse grid and QML evaluations on sparse grid from clc_intp_6f_sparse_data.ipynb

In [ ]:
# Importing necessary packages (from Lorenzos QML model)
import torch as torch
import numpy as np
import quasi_interpolation as qi
import sys
import os
import importlib
from sklearn.metrics import r2_score,mean_squared_error




path_base = Path(os.getcwd()) 
# Current path for importing custom functions
sys.path.insert(0, str(path_base / "clc_functions")) 

import input_transform
importlib.reload(input_transform)
from input_transform import inputs_transform , inverse_transform_clc



In [ ]:
# Folder in which to find the test inputs
test_inputs_folder = 'test_data/'


transform_output = True
name_out_transf = ''
if transform_output:
    name_out_transf = '_transformedCLC'

### Interval within which transformed clc should be bounded
bound_output = [0.0, 1.0]

### Upper bound for input transformation
transform_input = True
upperbound = np.pi
upperbound_name = '1p0pi'

batch_size = 100
n_batch_name = str(batch_size)

# Learning rate
learning_rate = 0.001
learning_rate_name = '0p001'

### Kept features
features_kept = ['hus', 'clw', 'cli', 'ta', 'pa', 'hwind']
no_of_features = len(features_kept)

### Architecture specifications
no_qubits = no_of_features

ind_features = [0,1,2,3,4,6]

In [ ]:
## PQC architecture layout
# No of shots for circuit evaluation
no_shots = 1000 # or #inf
#No of shots that were used in training 
no_shots_training = 'inf'
#Architecture to be used 'ZZXY' or 'XYZ'
name_arch = 'XYZ'
### PQC architecture layout + optimal params
if name_arch == 'XYZ':
    n_enc = 4;  n_dec = 2; 
    if no_shots_training == 'inf':
        best_exp = 1 
    else:
        best_exp = 4
elif name_arch == 'ZZXY':
    n_enc = 2;  n_dec = 5; # best_exp = 6 # for inifinite shots
    if no_shots_training == 'inf':
        best_exp = 6 
    else:
        best_exp = 3

n_enc_name = str(n_enc)
n_dec_name = str(n_dec)

# Load optimal params:
if no_shots_training == 'inf':
    params_folder = 'optimal_params/'
    namefilepars = 'optimal_params'
    name_end = ('_upperbound' + upperbound_name + name_out_transf + '_' + name_arch + '_Nenc' + n_enc_name + 
                '_Ndec' + n_dec_name + '_batch' + n_batch_name + '_lr' + learning_rate_name + '_test' + str(best_exp))
    filename_pars = namefilepars + name_end + '.npy'
else: #varreg, trainign with 1000t
    params_folder = 'optimal_params/'
    filename_pars = 'optimal_params_Nshots'+str(no_shots_training)+'_multinomial_transformedCLC_' + name_arch + '_Nenc' +str(n_enc) + '_Ndec' + str(n_dec) + '_batch100_alpha0p005_iniparam12345_test' + str(best_exp) +'.npy'
    

path_file = os.path.join(params_folder, filename_pars)
opt_params_np = np.load(path_file)
opt_params = jnp.asarray(opt_params_np)


In [ ]:

### ---------------------------------------------------------------------------------------- ###
## ----------------------------------- Load testing data ------------------------------------ ##
### ---------------------------------------------------------------------------------------- ###

namefilein = 'cirrus_inputs_raw_8features.npy'
namefileout = 'cirrus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cirrus_full = np.load(path_file)
test_inputs_cirrus = test_inputs_cirrus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cirrus = np.load(path_file)
no_testing_data_cirrus = test_inputs_cirrus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cirrus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cirrus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cirrus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cirrus = np.where(III_cirrus)[0]
no_test_samples_to_evaluate_cirrus = np.sum(III_cirrus)
print('No. test samples to evaluate (cirrus): ', no_test_samples_to_evaluate_cirrus)

namefilein = 'cumulus_inputs_raw_8features.npy'
namefileout = 'cumulus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cumulus_full = np.load(path_file)
test_inputs_cumulus = test_inputs_cumulus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cumulus = np.load(path_file)
no_testing_data_cumulus = test_inputs_cumulus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cumulus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cumulus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cumulus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cumulus = np.where(III_cumulus)[0]
no_test_samples_to_evaluate_cumulus = np.sum(III_cumulus)
print('No. test samples to evaluate (cumulus): ', no_test_samples_to_evaluate_cumulus)

namefilein = 'deepconv_inputs_raw_8features.npy'
namefileout = 'deepconv_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_deepconv_full = np.load(path_file)
test_inputs_deepconv = test_inputs_deepconv_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_deepconv = np.load(path_file)
no_testing_data_deepconv = test_inputs_deepconv.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_deepconv[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_deepconv[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_deepconv = np.squeeze(np.logical_not(IIIlowclt))
indsIII_deepconv = np.where(III_deepconv)[0]
no_test_samples_to_evaluate_deepconv = np.sum(III_deepconv)
print('No. test samples to evaluate (deepconv): ', no_test_samples_to_evaluate_deepconv)

namefilein = 'stratus_inputs_raw_8features.npy'
namefileout = 'stratus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_stratus_full = np.load(path_file)
test_inputs_stratus = test_inputs_stratus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_stratus = np.load(path_file)
no_testing_data_stratus = test_inputs_stratus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_stratus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_stratus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_stratus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_stratus = np.where(III_stratus)[0]
no_test_samples_to_evaluate_stratus = np.sum(III_stratus)
print('No. test samples to evaluate (stratus): ', no_test_samples_to_evaluate_stratus)

### Transform inputs if needed
if transform_input:
    bound_input = [0.0, upperbound]
    bounds = [bound_input for _ in features_kept]
    test_inputs_cirrus_t = inputs_transform(test_inputs_cirrus, features_kept, bounds)
    test_inputs_cumulus_t = inputs_transform(test_inputs_cumulus, features_kept, bounds)
    test_inputs_deepconv_t = inputs_transform(test_inputs_deepconv, features_kept, bounds)
    test_inputs_stratus_t = inputs_transform(test_inputs_stratus, features_kept, bounds)

### Convert testing data to jax numpy arrays
jnp_test_inputs_cirrus = np.asarray(test_inputs_cirrus_t)
jnp_test_inputs_cumulus = np.asarray(test_inputs_cumulus_t)
jnp_test_inputs_deepconv = np.asarray(test_inputs_deepconv_t)
jnp_test_inputs_stratus = np.asarray(test_inputs_stratus_t)


#test_data = np.concatenate([jnp_test_inputs_cirrus[indsIII_cirrus[0:1000],:], jnp_test_inputs_cumulus[indsIII_cumulus[0:1000],:],jnp_test_inputs_stratus[indsIII_stratus[0:1000],:],jnp_test_inputs_deepconv[indsIII_deepconv[0:1000],:]], axis = 0)
#test_output =  np.concatenate([test_outputs_cirrus[indsIII_cirrus[0:1000]], test_outputs_cumulus[indsIII_cumulus[0:1000]],test_outputs_stratus[indsIII_stratus[0:1000]],test_outputs_deepconv[indsIII_deepconv[0:1000]]], axis = 0)


test_data = jnp_test_inputs_cirrus[indsIII_cirrus[0:1000],:]
test_output =  test_outputs_cirrus[indsIII_cirrus[0:1000]]


In [ ]:

def post_processing(outsbatch):
    outsbatch = np.asarray(outsbatch)
    outsbatch = np.squeeze(outsbatch)
    outsbatch = np.minimum(outsbatch, 1.0)
    outsbatch = np.maximum(outsbatch, 0.0)
    outsbatch = inverse_transform_clc(outsbatch)
    return outsbatch


# Load coordinates for sparse grid and functions values to evaluate

In [ ]:


L = 7
d = 6
sparse_grid = np.load('sparse_grid_coo_'+str(L)+'.npy')
print(sparse_grid.shape)
Y_data = np.load('sparse_grid_val_'+name_arch+'_'+str(L)+'.npy')


# Define interpolant

In [ ]:

# Full grid tensor:
full_grid_shape = (2**(L)+1,)*d
indices = (2**L)*sparse_grid

coords = torch.tensor(indices.T, dtype=torch.long)

values = torch.tensor(range(1,indices.shape[0]+1))

# Create a sparse tensor
sparse_tensor = torch.sparse_coo_tensor(coords, values, size=full_grid_shape)
sparse_tensor = sparse_tensor.coalesce()

print(sparse_tensor.shape)


In [ ]:

ins_batch = 1/(2.*np.pi)*test_data
outs_batch= qi.fun_interpolant_sparse(ins_batch,L,d, sparse_tensor ,sparse_grid,Y_data)
outs_batch = post_processing(outs_batch)
pred_test_outputs_cs = np.squeeze(outs_batch)

print('done')

In [ ]:

print('--') 
print('Error to test data')

mse = mean_squared_error(test_output,pred_test_outputs_cs)
r2 = r2_score(test_output,pred_test_outputs_cs)

print(mse)


In [ ]:
#print(outs_batch)
result = {'L': L, 'input': ins_batch, 'output':test_data , 
          'pred_output': pred_test_outputs_cs, 'qi_setting': 'sparse_grid, RBF Kernel', 
          'arch': name_arch,'mse':mse}
np.save('result_sparse_qi_'+name_arch + '_'+str(L)+'N' + str(len(test_output)),result)